# Baseline vs ABS — Analysis

This notebook visualizes the results produced by `python -m src.pipeline`.
Run the pipeline first; this notebook only consumes `output/eval_results.csv`
and `output/descriptions/*.json`.

## Outline
1. Load eval table
2. Color-word ratio: distribution + paired test
3. Length / latency / cost comparison
4. Per-genre breakdown
5. Qualitative side-by-side examples

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

EVAL_CSV = REPO_ROOT / 'output' / 'eval_results.csv'
DESC_DIR = REPO_ROOT / 'output' / 'descriptions'

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv(EVAL_CSV)
df.head()

## 1. Color-word ratio distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x='condition', y='color_word_ratio', ax=ax, order=['baseline', 'abs'])
sns.stripplot(data=df, x='condition', y='color_word_ratio', ax=ax,
              order=['baseline', 'abs'], color='black', alpha=0.4, size=3)
ax.set_title('Color-word ratio per description (lower = better for ABS)')
ax.set_ylabel('color words / total tokens')
plt.tight_layout()

## 2. Paired test (Wilcoxon + paired t)

Both conditions are computed for the same set of artworks, so we use a paired test.
Wilcoxon signed-rank does not assume normality; we also report a paired t-test
for reference.

In [ ]:
pivot = df.pivot_table(
    index='artwork_id', columns='condition', values='color_word_ratio'
).dropna()
print(f'Paired N = {len(pivot)}')
if len(pivot) >= 2:
    w_stat, w_p = stats.wilcoxon(pivot['baseline'], pivot['abs'])
    t_stat, t_p = stats.ttest_rel(pivot['baseline'], pivot['abs'])
    print(f'Wilcoxon signed-rank: stat={w_stat:.3f}, p={w_p:.3e}')
    print(f'Paired t-test:        t={t_stat:.3f}, p={t_p:.3e}')
    diff = pivot['baseline'] - pivot['abs']
    print(f'\nMean difference (baseline - abs): {diff.mean():.4f}')
    print(f'95% CI: [{diff.mean() - 1.96*diff.std()/np.sqrt(len(diff)):.4f}, '
          f'{diff.mean() + 1.96*diff.std()/np.sqrt(len(diff)):.4f}]')

## 3. Length / latency / cost

In [ ]:
agg = df.groupby('condition').agg(
    n=('artwork_id', 'count'),
    mean_chars=('char_count', 'mean'),
    mean_sentences=('sentence_count', 'mean'),
    mean_latency=('latency_sec', 'mean'),
    mean_cost=('cost_usd', 'mean'),
    total_cost=('cost_usd', 'sum'),
).round(3)
agg

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, title in zip(
    axes,
    ['char_count', 'latency_sec', 'cost_usd'],
    ['Length (chars)', 'Latency (s)', 'Cost (USD)']
):
    sns.boxplot(data=df, x='condition', y=col, ax=ax, order=['baseline', 'abs'])
    ax.set_title(title)
plt.tight_layout()

## 4. Per-genre breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.boxplot(data=df, x='genre', y='color_word_ratio', hue='condition',
            ax=ax, hue_order=['baseline', 'abs'])
ax.set_title('Color-word ratio by genre')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()

In [ ]:
df.groupby(['genre', 'condition'])['color_word_ratio'].mean().unstack().round(4)

## 5. Qualitative examples

Five randomly chosen artworks shown side-by-side. The ABS column should
have effectively zero color words while remaining spatial-first and structured.

In [ ]:
def load_descriptions():
    out = {}
    for p in DESC_DIR.glob('*.json'):
        rec = json.loads(p.read_text(encoding='utf-8'))
        out.setdefault(rec['artwork_id'], {})[rec['condition']] = rec
    return out

descs = load_descriptions()
ids_both = [aid for aid, d in descs.items() if {'baseline', 'abs'} <= set(d)]
rng = np.random.default_rng(0)
picked = rng.choice(ids_both, size=min(5, len(ids_both)), replace=False)

for aid in picked:
    base = descs[aid]['baseline']
    abs_ = descs[aid]['abs']
    print('=' * 80)
    print(f'{aid}  ({base["artist"]} / {base["genre"]})')
    print('-' * 80)
    print('[BASELINE]')
    print(base['description'])
    print()
    print('[ABS]')
    print(abs_['description'])
    print()